# ROST model statistics from abundance data

This notebook shows a generic, domain-neutral workflow for fitting `rostpy` to an abundance table and extracting model statistics.

The important modeling point is that ROST is stochastic: the same model configuration can give different topic assignments each time. For that reason, this example fits the same model multiple times and keeps the best run according to transparent summary statistics.

The notebook covers:

- deterministic abundance-to-count scaling;
- fitting one ROST model;
- extracting sample-topic probabilities;
- extracting topic-word probabilities;
- computing perplexity, topic entropy, topic similarity, and UMass-style coherence;
- repeating the same stochastic model multiple times;
- selecting the best run.

In [1]:
import itertools

import numpy as np
import pandas as pd
import rostpy

## 1. Create an example abundance table

Rows are samples. Columns are words/features/taxa. Values are abundance-like measurements.

The example is deliberately small, but it has two broad patterns: samples 1–3 are dominated by `word_A`/`word_B`, while samples 4–6 are dominated by `word_D`/`word_E`.

In [2]:
abundance = pd.DataFrame(
    {
        "word_A": [2500, 2200, 1800, 100, 50, 100],
        "word_B": [1800, 1500, 1700, 200, 100, 150],
        "word_C": [400, 300, 500, 600, 500, 400],
        "word_D": [100, 100, 200, 1800, 2200, 2000],
        "word_E": [50, 100, 150, 1600, 1900, 2100],
    },
    index=[f"sample_{i:03d}" for i in range(1, 7)],
)

taxa = abundance.columns.to_list()
sample_ids = abundance.index.to_list()

abundance

,word_A,word_B,word_C,word_D,word_E
sample_001,2500,1800,400,100,50
sample_002,2200,1500,300,100,100
sample_003,1800,1700,500,200,150
sample_004,100,200,600,1800,1600
sample_005,50,100,500,2200,1900
sample_006,100,150,400,2000,2100


Expected output:

- `abundance.shape == (6, 5)`;
- six samples;
- five words/features/taxa.

## 2. Convert abundance to deterministic integer counts

ROST expects repeated integer word IDs, so abundance values must be represented as integer counts.

This example uses deterministic scaled counts, not fixed-depth relative tokenization. That preserves absolute abundance differences among samples while keeping the token lists small enough to model conveniently.

In [3]:
scale = 100

# round() avoids systematically flooring near-threshold values to zero.
counts = abundance.div(scale).round().astype(int)

token_counts = counts.sum(axis=1)

print("count table shape:", counts.shape)
print("token counts per sample:")
print(token_counts)
print("empty samples:", int((token_counts == 0).sum()))

counts

count table shape: (6, 5)
token counts per sample:
sample_001    48
sample_002    42
sample_003    44
sample_004    43
sample_005    47
sample_006    48
dtype: int64
empty samples: 0


,word_A,word_B,word_C,word_D,word_E
sample_001,25,18,4,1,0
sample_002,22,15,3,1,1
sample_003,18,17,5,2,2
sample_004,1,2,6,18,16
sample_005,0,1,5,22,19
sample_006,1,2,4,20,21


Expected output:

- `counts` is the integer table used to build ROST observations;
- `token_counts` is the number of word tokens each sample contributes;
- `empty samples` should usually be zero for a useful model.

## 3. Helper functions

These helpers keep the modeling workflow explicit:

- `make_observations` converts integer count rows to repeated word IDs;
- `fit_rost_once` fits one stochastic ROST model;
- `sample_topic_probabilities` converts word-level topic labels to sample-level topic probabilities;
- `topic_word_probabilities` normalizes the topic-word matrix;
- the remaining functions compute generic model statistics.

In [4]:
def make_observations(counts: pd.DataFrame) -> dict[int, list[int]]:
    """Convert each sample row into repeated integer word IDs."""
    observations = {}

    for t, (_, row) in enumerate(counts.iterrows()):
        obs = []
        for word_id, count in enumerate(row):
            obs.extend([word_id] * int(count))

        if obs:
            observations[t] = obs

    return observations


def fit_rost_once(
    counts: pd.DataFrame,
    K: int,
    n_epochs: int,
    nt: int = 1,
    alpha: float = 0.5,
    beta: float = 0.5,
    gamma: float = 0.1,
) -> tuple[rostpy.ROST_t, dict[int, list[int]]]:
    """Fit one stochastic ROST_t model from an integer count table."""
    observations = make_observations(counts)

    model = rostpy.ROST_t(
        V=counts.shape[1],
        K=K,
        alpha=alpha,
        beta=beta,
        gamma=gamma,
    )

    for t, obs in observations.items():
        model.add_observation([t], obs)

    for _ in range(n_epochs):
        rostpy.parallel_refine(model, nt)

    return model, observations


def sample_topic_probabilities(
    model: rostpy.ROST_t,
    observations: dict[int, list[int]],
    sample_ids: list[str],
) -> pd.DataFrame:
    """Return sample-topic probabilities from maximum-likelihood topic labels."""
    topic_counts = pd.DataFrame(0, index=sample_ids, columns=[f"topic_{k + 1}" for k in range(model.K)])

    for t, obs in observations.items():
        topics = model.get_ml_topics_for_pose([t])
        sample_id = sample_ids[t]

        for topic in topics:
            topic_counts.loc[sample_id, f"topic_{topic + 1}"] += 1

    return topic_counts.div(topic_counts.sum(axis=1), axis=0).fillna(0)


def topic_word_probabilities(model: rostpy.ROST_t, words: list[str]) -> pd.DataFrame:
    """Return normalized topic-word probabilities."""
    topic_word_counts = pd.DataFrame(
        model.get_topic_model(),
        index=[f"topic_{k + 1}" for k in range(model.K)],
        columns=words,
    )

    return topic_word_counts.div(topic_word_counts.sum(axis=1), axis=0).fillna(0)


def topic_entropy(topic_word_prob_df: pd.DataFrame) -> pd.Series:
    """Return Shannon entropy for each topic-word distribution."""
    p = topic_word_prob_df.to_numpy(dtype=float)
    entropy = -(p * np.log(p + 1e-12)).sum(axis=1)
    return pd.Series(entropy, index=topic_word_prob_df.index, name="entropy")


def mean_pairwise_topic_similarity(topic_word_prob_df: pd.DataFrame) -> float:
    """Return mean pairwise cosine similarity among topics."""
    x = topic_word_prob_df.to_numpy(dtype=float)

    if len(x) < 2:
        return np.nan

    similarities = []
    for i, j in itertools.combinations(range(len(x)), 2):
        denom = np.linalg.norm(x[i]) * np.linalg.norm(x[j])
        similarities.append(float(np.dot(x[i], x[j]) / denom) if denom else np.nan)

    return float(np.nanmean(similarities))


def umass_coherence(
    topic_word_prob_df: pd.DataFrame,
    counts: pd.DataFrame,
    top_n: int = 5,
) -> pd.Series:
    """Compute a simple UMass-style coherence score for each topic."""
    presence = counts.gt(0)
    scores = {}

    for topic, probs in topic_word_prob_df.iterrows():
        top_words = probs.sort_values(ascending=False).head(top_n).index.to_list()
        terms = []

        for i in range(1, len(top_words)):
            for j in range(i):
                wi = top_words[i]
                wj = top_words[j]
                d_wi_wj = int((presence[wi] & presence[wj]).sum())
                d_wj = int(presence[wj].sum())
                terms.append(np.log((d_wi_wj + 1) / max(d_wj, 1)))

        scores[topic] = float(np.mean(terms)) if terms else np.nan

    return pd.Series(scores, name="umass_coherence")


def summarize_model(
    model: rostpy.ROST_t,
    observations: dict[int, list[int]],
    counts: pd.DataFrame,
    words: list[str],
    sample_ids: list[str],
) -> dict:
    """Extract generic model statistics from one fitted ROST model."""
    sample_topic_df = sample_topic_probabilities(model, observations, sample_ids)
    topic_word_df = topic_word_probabilities(model, words)
    entropy = topic_entropy(topic_word_df)
    coherence = umass_coherence(topic_word_df, counts)

    return {
        "perplexity": model.perplexity(),
        "num_cells": model.num_cells(),
        "refine_count": model.get_refine_count(),
        "word_refine_count": model.get_word_refine_count(),
        "mean_topic_entropy": float(entropy.mean()),
        "mean_umass_coherence": float(coherence.mean()),
        "mean_topic_similarity": mean_pairwise_topic_similarity(topic_word_df),
        "sample_topic_df": sample_topic_df,
        "topic_word_df": topic_word_df,
        "topic_entropy": entropy,
        "topic_coherence": coherence,
    }

## 4. Fit one model and inspect outputs

This section fits one model so the intermediate outputs are easy to inspect before running repeated stochastic fits.

In [5]:
model, observations = fit_rost_once(counts, K=2, n_epochs=50, nt=1)
stats = summarize_model(model, observations, counts, taxa, sample_ids)

print("perplexity:", stats["perplexity"])
print("number of cells:", stats["num_cells"])
print("cell refinements:", stats["refine_count"])
print("word refinements:", stats["word_refine_count"])

perplexity: 1.0
number of cells: 6
cell refinements: 300
word refinements: 13600


Expected output:

- `perplexity` is a positive model-fit diagnostic;
- `number of cells` should equal the number of non-empty samples;
- `cell refinements` should increase with `n_epochs`;
- `word refinements` should increase with the number of tokens refined.

In [6]:
# Sample-topic probabilities: rows are samples, columns are topics.
stats["sample_topic_df"]

,topic_1,topic_2
sample_001,0.000000,1.000000
sample_002,0.023810,0.976190
sample_003,0.090909,0.909091
sample_004,0.930233,0.069767
sample_005,1.000000,0.000000
sample_006,0.979167,0.020833


In [7]:
# Topic-word probabilities: rows are topics, columns are words/features/taxa.
stats["topic_word_df"]

,word_A,word_B,word_C,word_D,word_E
topic_1,0.0,0.028986,0.101449,0.449275,0.420290
topic_2,0.5,0.380597,0.097015,0.014925,0.007463


In [8]:
# Topic-specific entropy and coherence summaries.
pd.concat([stats["topic_entropy"], stats["topic_coherence"]], axis=1)

,entropy,umass_coherence
topic_1,1.058558,0.082710
topic_2,1.039869,0.100942


## 5. Repeat stochastic fits and keep the best run

Because ROST is stochastic, a single fit is not enough for a stable workflow. This section runs the same model configuration multiple times.

The example selection score combines three ranks:

- lower perplexity is better;
- higher UMass coherence is better;
- lower mean topic similarity is better because it means topics are less redundant.

You can change this selection rule depending on whether predictive fit, interpretability, or topic separation is most important.

In [9]:
K = 2
n_runs = 10
n_epochs = 50

run_records = []
run_outputs = {}

for run_id in range(n_runs):
    model, observations = fit_rost_once(counts, K=K, n_epochs=n_epochs, nt=1)
    stats = summarize_model(model, observations, counts, taxa, sample_ids)

    run_records.append(
        {
            "run_id": run_id,
            "perplexity": stats["perplexity"],
            "mean_umass_coherence": stats["mean_umass_coherence"],
            "mean_topic_entropy": stats["mean_topic_entropy"],
            "mean_topic_similarity": stats["mean_topic_similarity"],
            "refine_count": stats["refine_count"],
            "word_refine_count": stats["word_refine_count"],
        }
    )

    run_outputs[run_id] = {
        "model": model,
        "observations": observations,
        "stats": stats,
    }

run_summary_df = pd.DataFrame(run_records)

# Rank each run. Lower rank is better.
run_summary_df["perplexity_rank"] = run_summary_df["perplexity"].rank(method="min", ascending=True)
run_summary_df["coherence_rank"] = run_summary_df["mean_umass_coherence"].rank(method="min", ascending=False)
run_summary_df["similarity_rank"] = run_summary_df["mean_topic_similarity"].rank(method="min", ascending=True)

run_summary_df["selection_score"] = (
    run_summary_df["perplexity_rank"]
    + run_summary_df["coherence_rank"]
    + run_summary_df["similarity_rank"]
)

run_summary_df = run_summary_df.sort_values("selection_score").reset_index(drop=True)
run_summary_df

,run_id,perplexity,mean_umass_coherence,mean_topic_entropy,mean_topic_similarity,refine_count,word_refine_count,perplexity_rank,coherence_rank,similarity_rank,selection_score
0,7,1.0,0.100942,0.945165,0.023551,300,13600,1.0,1.0,1.0,3.0
1,8,1.0,0.100942,0.979549,0.038194,300,13600,1.0,1.0,2.0,4.0
2,4,1.0,0.100942,0.972948,0.038615,300,13600,1.0,1.0,3.0,5.0
3,6,1.0,0.100942,0.977757,0.040244,300,13600,1.0,1.0,4.0,6.0
4,5,1.0,0.100942,1.009330,0.053290,300,13600,1.0,1.0,7.0,9.0
5,0,1.0,0.100942,1.013317,0.056079,300,13600,1.0,1.0,8.0,10.0
6,9,1.0,0.100942,1.031495,0.068119,300,13600,1.0,1.0,9.0,11.0
7,2,1.0,0.091826,0.994082,0.046028,300,13600,1.0,8.0,5.0,14.0
8,3,1.0,0.091826,1.003115,0.051481,300,13600,1.0,8.0,6.0,15.0
9,1,1.0,0.091826,1.040152,0.076641,300,13600,1.0,8.0,10.0,19.0


Expected output:

A run-summary table with one row per stochastic fit. The best run is the first row after sorting by `selection_score`.

Interpretation:

- lower `perplexity` indicates better predictive fit;
- higher `mean_umass_coherence` indicates more coherent top words;
- lower `mean_topic_similarity` indicates less redundant topics;
- `selection_score` is an editable, transparent way to combine these criteria.

In [10]:
best_run_id = int(run_summary_df.iloc[0]["run_id"])
best = run_outputs[best_run_id]

print("best run:", best_run_id)
print("best run perplexity:", best["stats"]["perplexity"])
print("best run mean coherence:", best["stats"]["mean_umass_coherence"])
print("best run mean topic similarity:", best["stats"]["mean_topic_similarity"])

best run: 7
best run perplexity: 1.0
best run mean coherence: 0.10094167098636389
best run mean topic similarity: 0.0235511239163639


## 6. Inspect the best run

After choosing the best stochastic run, inspect its sample-topic probabilities, topic-word probabilities, and topic-level statistics.

In [11]:
best["stats"]["sample_topic_df"]

,topic_1,topic_2
sample_001,0.020833,0.979167
sample_002,0.047619,0.952381
sample_003,0.090909,0.909091
sample_004,0.930233,0.069767
sample_005,0.978723,0.021277
sample_006,0.937500,0.062500


In [12]:
best["stats"]["topic_word_df"]

,word_A,word_B,word_C,word_D,word_E
topic_1,0.000000,0.000000,0.102190,0.467153,0.430657
topic_2,0.496296,0.407407,0.096296,0.000000,0.000000


In [13]:
pd.concat(
    [
        best["stats"]["topic_entropy"],
        best["stats"]["topic_coherence"],
    ],
    axis=1,
)

,entropy,umass_coherence
topic_1,0.951441,0.100942
topic_2,0.938889,0.100942
